# Pale vs DVC — real comparison

Compares Pale and DVC on three transfer learning scenarios using a pretrained
ResNet-18 backbone. DVC is used directly (not simulated) — checkpoints are
tracked with `dvc add` and pushed to a local DVC remote. Storage is measured
as actual bytes on disk in the DVC cache and the Pale CAS store.

**Scenarios:**

1. **Hyperparameter sweep** — 8 fine-tuning runs from the same pretrained base,
   varying learning rate and weight decay. The frozen backbone is identical
   across all runs.

2. **Multi-seed sweep** — 4 runs with the same hyperparameters but different
   random seeds. Tests whether Pale's cross-run savings hold when the only
   difference between runs is initialization noise.

3. **Mid-training resume** — A single run saved at epoch 5, then resumed and
   continued to epoch 10. Tests Pale's `parent_step` lineage tracking vs
   DVC's file-per-checkpoint approach.

**DVC setup:** local remote at `/tmp/pale_dvc_remote`. Each checkpoint is
tracked with `dvc add` + `dvc push`. DVC cache size is measured as the sum
of all files under `.dvc/cache` after all pushes.

**Pale setup:** shared `PaleStore` root for all runs within each scenario.
Pale bytes = sum of `.chunk` file sizes under `{root}/objects/`.

Both sides use zstd level 3 compression.

In [ ]:
!pip install -q git+https://github.com/Olamyy/pale.git@hash-cache-no-op-path zstandard torch torchvision dvc

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, str(Path(".").resolve()))
from utils import pale_bytes, _fmt_bytes

from pale.store import PaleStore
from pale.adapters.pytorch import PyTorchAdapter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print("Imports OK")

## Shared helpers

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

NUM_CLASSES = 10


def _make_model() -> nn.Module:
    """Pretrained ResNet-18 with frozen backbone, trainable FC only."""
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad_(False)
    return model.to(device)


def _make_loader(seed: int, n: int = 512):
    rng = np.random.default_rng(seed)
    X = torch.from_numpy(rng.standard_normal((n, 3, 32, 32)).astype(np.float32)).to(device)
    y = torch.from_numpy(rng.integers(0, NUM_CLASSES, n).astype(np.int64)).to(device)
    return torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X, y),
        batch_size=64, shuffle=True, num_workers=0,
    )


def train_epochs(
    model: nn.Module,
    loader,
    n_epochs: int,
    lr: float,
    weight_decay: float = 0.0,
) -> list[dict]:
    """Train for n_epochs. Returns one state dict per epoch."""
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=weight_decay,
    )
    criterion = nn.CrossEntropyLoss()
    model.train()
    state_dicts = []
    for _ in range(n_epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        state_dicts.append({k: v.clone().cpu() for k, v in model.state_dict().items()})
    return state_dicts


def _sd_to_model(sd: dict) -> nn.Module:
    m = _make_model()
    m.load_state_dict({k: v.clone() for k, v in sd.items()})
    return m


print("Model helpers OK")

In [ ]:
def dvc_init(repo_dir: Path, remote_dir: Path) -> None:
    """Initialise a DVC repo with a local remote."""
    repo_dir.mkdir(parents=True, exist_ok=True)
    remote_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.email", "bench@pale"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.name", "Pale Bench"], cwd=repo_dir, check=True)
    subprocess.run(["dvc", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(
        ["dvc", "remote", "add", "-d", "local", str(remote_dir)],
        cwd=repo_dir, check=True,
    )


def dvc_track_and_push(repo_dir: Path, checkpoint_path: Path) -> None:
    """Add a checkpoint file to DVC and push to the local remote."""
    subprocess.run(["dvc", "add", str(checkpoint_path)], cwd=repo_dir, check=True,
                   capture_output=True)
    subprocess.run(["dvc", "push"], cwd=repo_dir, check=True, capture_output=True)


def dvc_cache_bytes(remote_dir: Path) -> int:
    """Total bytes stored in the DVC local remote."""
    return sum(p.stat().st_size for p in remote_dir.rglob("*") if p.is_file())


def print_comparison(label: str, n_checkpoints: int, dvc_b: int, pale_b: int) -> None:
    savings = (dvc_b - pale_b) / dvc_b * 100 if dvc_b else 0
    print(f"\n{label}")
    print(f"  Checkpoints : {n_checkpoints}")
    print(f"  DVC         : {_fmt_bytes(dvc_b)}")
    print(f"  Pale        : {_fmt_bytes(pale_b)}")
    print(f"  Savings     : {savings:.1f}%")


print("DVC helpers OK")

---
## Scenario 1 — Hyperparameter sweep

8 fine-tuning runs from the same pretrained ResNet-18 base.
Varying learning rate (4 values) × weight decay (2 values).
10 epochs per run, 80 checkpoints total.

**Expected:** DVC stores 8 independent sets of 10 checkpoint files.
Pale stores the pretrained backbone once; each additional run costs only
FC weights + BN running stats (~53 KB).

In [ ]:
N_EPOCHS_S1 = 10

SWEEP_CONFIGS = [
    {"run_id": "lr1e-3_wd0",   "lr": 1e-3, "wd": 0.0,  "seed": 42},
    {"run_id": "lr1e-3_wd1e-4", "lr": 1e-3, "wd": 1e-4, "seed": 42},
    {"run_id": "lr5e-4_wd0",   "lr": 5e-4, "wd": 0.0,  "seed": 42},
    {"run_id": "lr5e-4_wd1e-4", "lr": 5e-4, "wd": 1e-4, "seed": 42},
    {"run_id": "lr2e-3_wd0",   "lr": 2e-3, "wd": 0.0,  "seed": 42},
    {"run_id": "lr2e-3_wd1e-4", "lr": 2e-3, "wd": 1e-4, "seed": 42},
    {"run_id": "lr1e-4_wd0",   "lr": 1e-4, "wd": 0.0,  "seed": 42},
    {"run_id": "lr1e-4_wd1e-4", "lr": 1e-4, "wd": 1e-4, "seed": 42},
]

print(f"Runs: {len(SWEEP_CONFIGS)}")
print(f"Total checkpoints: {len(SWEEP_CONFIGS) * N_EPOCHS_S1}")

In [ ]:
s1_state_dicts = {}

for cfg in SWEEP_CONFIGS:
    torch.manual_seed(cfg["seed"])
    model = _make_model()
    loader = _make_loader(cfg["seed"])
    t0 = time.time()
    print(f"  Training {cfg['run_id']}...", end=" ", flush=True)
    s1_state_dicts[cfg["run_id"]] = train_epochs(
        model, loader, N_EPOCHS_S1, cfg["lr"], cfg["wd"]
    )
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints: {sum(len(v) for v in s1_state_dicts.values())}")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo    = tmp / "dvc_repo"
    dvc_remote  = tmp / "dvc_remote"
    pale_root   = tmp / "pale"
    ckpt_dir    = dvc_repo / "checkpoints"

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    n_checkpoints = 0
    print("Saving checkpoints...")

    for cfg in SWEEP_CONFIGS:
        run_id = cfg["run_id"]
        run_ckpt_dir = ckpt_dir / run_id
        run_ckpt_dir.mkdir()

        # DVC: save each epoch as a .pt file, track and push
        for epoch, sd in enumerate(s1_state_dicts[run_id], 1):
            path = run_ckpt_dir / f"epoch_{epoch:02d}.pt"
            torch.save(sd, path)
            dvc_track_and_push(dvc_repo, path)
            n_checkpoints += 1

        # Pale: save all epochs for this run
        with PaleStore(root=pale_root, run_id=run_id, adapter=PyTorchAdapter()) as store:
            for epoch, sd in enumerate(s1_state_dicts[run_id], 1):
                store.save(_sd_to_model(sd), step=epoch)

        print(f"  {run_id}: done")

    dvc_b  = dvc_cache_bytes(dvc_remote)
    pale_b = pale_bytes(pale_root)

print_comparison("Scenario 1 — Hyperparameter sweep (8 runs × 10 epochs)",
                 n_checkpoints, dvc_b, pale_b)

# Per-run marginal cost
print("\nMarginal DVC cost per run (file-level, no cross-run dedup):")
print(f"  {_fmt_bytes(dvc_b)} / {len(SWEEP_CONFIGS)} runs = {_fmt_bytes(dvc_b // len(SWEEP_CONFIGS))} per run")

---
## Scenario 2 — Multi-seed sweep

4 runs with identical hyperparameters (lr=1e-3, wd=0), different seeds.
10 epochs per run, 40 checkpoints total.

This is the minimal reproducibility test: run the same experiment N times
to confirm variance. DVC treats each run as completely independent.
Pale shares the frozen backbone across all runs regardless.

**Expected:** Pale savings similar to scenario 1 — backbone stored once,
only FC + BN stats unique per run.

In [ ]:
N_EPOCHS_S2 = 10

SEED_CONFIGS = [
    {"run_id": "seed_42",  "seed": 42,  "lr": 1e-3, "wd": 0.0},
    {"run_id": "seed_99",  "seed": 99,  "lr": 1e-3, "wd": 0.0},
    {"run_id": "seed_7",   "seed": 7,   "lr": 1e-3, "wd": 0.0},
    {"run_id": "seed_123", "seed": 123, "lr": 1e-3, "wd": 0.0},
]

print(f"Runs: {len(SEED_CONFIGS)}")
print(f"Total checkpoints: {len(SEED_CONFIGS) * N_EPOCHS_S2}")

In [ ]:
s2_state_dicts = {}

for cfg in SEED_CONFIGS:
    torch.manual_seed(cfg["seed"])
    model = _make_model()
    loader = _make_loader(cfg["seed"])
    t0 = time.time()
    print(f"  Training {cfg['run_id']}...", end=" ", flush=True)
    s2_state_dicts[cfg["run_id"]] = train_epochs(
        model, loader, N_EPOCHS_S2, cfg["lr"], cfg["wd"]
    )
    print(f"{time.time() - t0:.1f}s")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    pale_root  = tmp / "pale"
    ckpt_dir   = dvc_repo / "checkpoints"

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    n_checkpoints = 0
    pale_size_after = {}

    for cfg in SEED_CONFIGS:
        run_id = cfg["run_id"]
        run_ckpt_dir = ckpt_dir / run_id
        run_ckpt_dir.mkdir()

        for epoch, sd in enumerate(s2_state_dicts[run_id], 1):
            path = run_ckpt_dir / f"epoch_{epoch:02d}.pt"
            torch.save(sd, path)
            dvc_track_and_push(dvc_repo, path)
            n_checkpoints += 1

        with PaleStore(root=pale_root, run_id=run_id, adapter=PyTorchAdapter()) as store:
            for epoch, sd in enumerate(s2_state_dicts[run_id], 1):
                store.save(_sd_to_model(sd), step=epoch)

        pale_size_after[run_id] = pale_bytes(pale_root)

    dvc_b  = dvc_cache_bytes(dvc_remote)
    pale_b = pale_bytes(pale_root)

print_comparison("Scenario 2 — Multi-seed sweep (4 runs × 10 epochs)",
                 n_checkpoints, dvc_b, pale_b)

# Show marginal Pale cost per run
print("\nPale store size after each run:")
prev = 0
for run_id, size in pale_size_after.items():
    print(f"  {run_id:<12} {_fmt_bytes(size):>10}  (+{_fmt_bytes(size - prev)})")
    prev = size

---
## Scenario 3 — Mid-training resume

A single fine-tuning run, saved at epoch 5, then resumed and continued
to epoch 10.

**DVC approach:** Two independent checkpoint files — `epoch_05.pt` and
`epoch_10.pt`. DVC tracks each as a separate file with no relationship
between them. Storage = compressed size of both files.

**Pale approach:** Step 5 is saved normally. Step 10 is saved with
`parent_step=5` — the manifest records the lineage. Only tensors that
changed between step 5 and step 10 are written as new chunks; unchanged
tensors (the entire frozen backbone) are referenced from step 5's manifest.

**Expected:** Pale stores the backbone once (in step 5). Step 10 writes
only the FC weights and BN running stats that changed during epochs 6–10.
DVC stores two complete copies of the full 45 MB state dict.

In [ ]:
torch.manual_seed(42)
model = _make_model()
loader = _make_loader(42)

# Phase 1: train epochs 1–5
print("Phase 1: training epochs 1–5...", end=" ", flush=True)
t0 = time.time()
phase1_sds = train_epochs(model, loader, n_epochs=5, lr=1e-3)
print(f"{time.time() - t0:.1f}s")

# Snapshot model state at epoch 5 for resume
sd_at_5 = phase1_sds[-1]

# Phase 2: resume from epoch 5, train epochs 6–10
# Load epoch-5 state into a fresh model to simulate resume from checkpoint
resume_model = _sd_to_model(sd_at_5)
print("Phase 2: resuming from epoch 5, training epochs 6–10...", end=" ", flush=True)
t0 = time.time()
phase2_sds = train_epochs(resume_model, loader, n_epochs=5, lr=1e-3)
print(f"{time.time() - t0:.1f}s")

sd_at_10 = phase2_sds[-1]
print("\nCheckpoints: epoch_05 and epoch_10")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    pale_root  = tmp / "pale"
    ckpt_dir   = dvc_repo / "checkpoints"

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    # --- DVC ---
    path_5  = ckpt_dir / "epoch_05.pt"
    path_10 = ckpt_dir / "epoch_10.pt"
    torch.save(sd_at_5,  path_5)
    torch.save(sd_at_10, path_10)
    dvc_track_and_push(dvc_repo, path_5)
    dvc_track_and_push(dvc_repo, path_10)
    dvc_b = dvc_cache_bytes(dvc_remote)

    # --- Pale ---
    with PaleStore(root=pale_root, run_id="resume_run", adapter=PyTorchAdapter()) as store:
        # Save all intermediate steps (epochs 1–5)
        for epoch, sd in enumerate(phase1_sds, 1):
            store.save(_sd_to_model(sd), step=epoch)
        # Save the resume checkpoint with parent_step=5
        for epoch, sd in enumerate(phase2_sds, 6):
            store.save(_sd_to_model(sd), step=epoch, parent_step=5)

    pale_b = pale_bytes(pale_root)

    # Pale: what does just the epoch-5 and epoch-10 manifest reference?
    tensors_5  = store.load_tensors(step=5)
    tensors_10 = store.load_tensors(step=10)
    shared = set(tensors_5.keys()) & set(tensors_10.keys())
    identical = sum(
        1 for k in shared
        if np.array_equal(tensors_5[k], tensors_10[k])
    )

raw_size = path_5.stat().st_size + path_10.stat().st_size

print("Scenario 3 — Mid-training resume")
print(f"  Raw size (2 × .pt files)          : {_fmt_bytes(raw_size)}")
print(f"  DVC cache (2 compressed files)    : {_fmt_bytes(dvc_b)}")
print(f"  Pale store (all 10 steps)         : {_fmt_bytes(pale_b)}")
print()
print(f"  Tensors in epoch-5 manifest       : {len(tensors_5)}")
print(f"  Tensors in epoch-10 manifest      : {len(tensors_10)}")
print(f"  Tensors identical (5 → 10)        : {identical} / {len(shared)}")
print(f"  Tensors changed (5 → 10)          : {len(shared) - identical}")
print()
savings = (dvc_b - pale_b) / dvc_b * 100 if dvc_b else 0
print(f"  DVC vs Pale savings               : {savings:.1f}%")

---
## Summary

| Scenario | Checkpoints | DVC | Pale | Savings |
|---|---|---|---|---|
| 1 — Hyperparameter sweep (8 runs × 10 epochs) | 80 | TBD | TBD | TBD |
| 2 — Multi-seed sweep (4 runs × 10 epochs) | 40 | TBD | TBD | TBD |
| 3 — Mid-training resume (1 run, 2 checkpoints) | 2 | TBD | TBD | TBD |

**What DVC cannot do:**
- Cross-run deduplication: each run's files are independent in the DVC cache
- Sub-file deduplication: a checkpoint file is stored in full even if 99% of its bytes are identical to another checkpoint
- Lineage-aware storage: DVC tracks `epoch_05.pt` and `epoch_10.pt` as unrelated files; Pale records that step 10 was resumed from step 5 and stores only the diff

**What DVC does that Pale doesn't (yet):**
- Git integration: `.dvc` pointer files are committed to git, giving a full audit trail of which model version was used for which experiment
- Remote storage: DVC pushes to S3, GCS, Azure out of the box
- Pipeline tracking: `dvc run` / `dvc repro` track the full pipeline (data → training → evaluation), not just checkpoints